# Tier 3 — Full reproduction from raw data

**This is the from-scratch path** and it needs external tools the other tiers avoid. It is written as a documented recipe — run the cells yourself once the prerequisites below are in place. It is **not** meant to be executed unattended (the full pipeline takes hours).

**Prerequisites:**

- **MMseqs2** (`mmseqs` on `PATH`) — all-vs-all melodic alignment.
- **NCBI BLAST+** (`blastp`) + a local protein database, and **MAFFT** (`mafft`) — only for the protein-conservation leg.
- Network access (UniProt / NCBI / AlphaFold DB) for the protein leg.
- The raw corpora (next cell). **Meertens** cannot be auto-fetched — it is form-gated (see `DATA.md`); the fetch script handles TheSession + Savage only.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find the Src/thesession package."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "Src" / "thesession").is_dir():
            return cand
    raise RuntimeError("Could not locate the repo root (Src/thesession not found).")


REPO = find_repo_root()
if str(REPO / "Src") not in sys.path:
    sys.path.insert(0, str(REPO / "Src"))
print("Repo root:", REPO)


Repo root: /home/jmcbride/projects/TheSessionEvo


## 1. Fetch the raw corpora (pinned to the exact versions used in the paper)

```bash
python scripts/fetch_data.py
```

This clones TheSession-data at commit `4f2d9d5` and downloads the Savage Excel sheets at commit `82a8625`, placing them under `Data/`. Obtain Meertens separately via the form linked in `DATA.md` and place it under `Data/Meertens/`.

In [2]:
# Uncomment to run the fetch from within the notebook:
# import subprocess
# subprocess.run([sys.executable, str(REPO / "scripts" / "fetch_data.py")], check=True)

## 2. Check the external binaries are on PATH

`mmseqs` is required for the melodic pipeline; `blastp` + `mafft` only for the protein leg (its outputs are shipped in Tier 2, so you can skip it and keep the cached `ProteinData/` instead).

In [3]:
import shutil
for tool in ["mmseqs", "blastp", "mafft"]:
    path = shutil.which(tool)
    print(f"{tool:8} {'found: ' + path if path else 'NOT FOUND'}")

mmseqs   found: /home/jmcbride/projects/Tools/mmseqs/bin/mmseqs
blastp   found: /usr/bin/blastp
mafft    found: /usr/bin/mafft


## 3. Run the whole pipeline from scratch

`main(redo=True)` invalidates every cache and rebuilds everything: parse the corpora → extract parts → run MMseqs2 all-vs-all → parameter optimisation → all figure data → render figures. **This takes hours** and rewrites `Cache/`, `MMseqs/`, `FigureData/`, `ProteinData/`, and `Figures/`.

Use `redo=False` instead to fill in only what is missing (much faster if some caches already exist).

In [4]:
# Uncomment to run the full pipeline (hours):
# import matplotlib; matplotlib.use("Agg")
# import run_pipeline
# run_pipeline.main(redo=True)

When it finishes, `Figures/` holds the regenerated PDFs/PNGs, and `Cache/` + `MMseqs/` hold the intermediates that Tier 2 would otherwise have shipped — i.e. you have just rebuilt the Tier 1 and Tier 2 archives from raw inputs. To repackage them, run `scripts/build_zenodo.sh`.